# CNN-Transformer Hyperparameter Tuning

Focused tuning of CNN-Transformer for single-lead ECG MI detection.
Uses Optuna with two pruning strategies (MedianPruner vs HyperbandPruner)
to search over learning rate, warmup, and CNN freezing.

### Search space:
- `lr`: 1e-5 to 1e-2 (log scale)
- `warmup_epochs`: 0 to 5
- `freeze_cnn`: True or False

### Training:
- Pre-train: up to 100 epochs, patience 15
- Fine-tune: up to 50 epochs, patience 10
- Pruning kills unpromising trials early

### Tracking:
- DuckDB for experiment logging
- Best model saved to checkpoints/

In [1]:
import os
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split
import optuna
from optuna.pruners import MedianPruner, HyperbandPruner
import wfdb
import ast
import duckdb
from datetime import datetime

device = torch.device("mps") if torch.backends.mps.is_available() else torch.device("cpu")
print(f"Using device: {device}")

os.makedirs("../checkpoints", exist_ok=True)

db = duckdb.connect("../experiment_results.duckdb")
db.execute("""
    CREATE TABLE IF NOT EXISTS catnet_tuning (
        study_name VARCHAR, trial_number INTEGER, timestamp TIMESTAMP,
        pruner VARCHAR, lr DOUBLE, warmup_epochs INTEGER, freeze_cnn BOOLEAN,
        pretrain_best_auc DOUBLE, pretrain_epochs_run INTEGER,
        zeroshot_auc DOUBLE,
        finetune_best_auc DOUBLE, finetune_epochs_run INTEGER,
        status VARCHAR
    )
""")
print("DuckDB ready.")

Using device: mps
DuckDB ready.


## 1. Data Loading

In [2]:
# PTB-XL
ptb_path = '../data/ptb-xl-a-large-publicly-available-electrocardiography-dataset-1.0.3/'

def load_ptbxl_data(path):
    df = pd.read_csv(os.path.join(path, 'ptbxl_database.csv'), index_col=0)
    df.scp_codes = df.scp_codes.apply(lambda x: ast.literal_eval(x))
    return df

def aggregate_diagnostic(y_dic, agg_df):
    tmp = []
    for key in y_dic.keys():
        if key in agg_df.index: tmp.append(agg_df.loc[key].diagnostic_class)
    return list(set(tmp))

agg_df = pd.read_csv(os.path.join(ptb_path, 'scp_statements.csv'), index_col=0)
agg_df = agg_df[agg_df.diagnostic == 1]
df = load_ptbxl_data(ptb_path)
df['diagnostic_class'] = df.scp_codes.apply(aggregate_diagnostic, agg_df=agg_df)
df['MI'] = df.diagnostic_class.apply(lambda x: 1 if 'MI' in x else 0)

train_df = df[df.strat_fold < 9]
val_df = df[df.strat_fold == 9]
print(f"PTB-XL — Train: {len(train_df)} | Val: {len(val_df)} | MI: {df.MI.mean()*100:.2f}%")

# Shaoxing (full)
shaoxing_path = '../data/a-large-scale-12-lead-electrocardiogram-database-for-arrhythmia-study-1.0.0/'
MI_CODE = '164865005'

class ShaoxingDataset(Dataset):
    def __init__(self, base_path, lead_idx=0):
        self.base_path = base_path
        self.lead_idx = lead_idx
        self.records = []
        with open(os.path.join(base_path, 'RECORDS'), 'r') as f:
            subfolders = [line.strip() for line in f if line.strip()]
        for sub in subfolders:
            sub_path = os.path.join(base_path, sub)
            if os.path.exists(sub_path):
                for f_name in os.listdir(sub_path):
                    if f_name.endswith('.hea'):
                        self.records.append(os.path.join(sub, f_name.replace('.hea', '')))
        self.labels = []
        print(f"Scanning labels for {len(self.records)} Shaoxing records...")
        for rec in tqdm(self.records, desc="Labels"):
            try:
                _, meta = wfdb.rdsamp(os.path.join(base_path, rec))
                dx_list = []
                for c in meta['comments']:
                    if 'Dx:' in c:
                        dx_list = [x.strip() for x in c.split('Dx:')[1].strip().split(',')]
                        break
                self.labels.append(1 if MI_CODE in dx_list else 0)
            except: self.labels.append(0)
        print(f"MI: {sum(self.labels)} ({sum(self.labels)/len(self.labels)*100:.2f}%)")
    def __len__(self): return len(self.records)
    def __getitem__(self, idx):
        try:
            signal, _ = wfdb.rdsamp(os.path.join(self.base_path, self.records[idx]))
            s = signal[::5, self.lead_idx].astype(np.float32)
            s = (s - s.mean()) / (s.std() + 1e-8)
            return torch.tensor(s, dtype=torch.float32).unsqueeze(0), torch.tensor(self.labels[idx], dtype=torch.float32)
        except:
            return torch.zeros(1, 1000), torch.tensor(0, dtype=torch.float32)

shaoxing_full = ShaoxingDataset(shaoxing_path)

# Stratified 40/60 split
shao_labels = np.array(shaoxing_full.labels)
train_idx, test_idx = train_test_split(np.arange(len(shaoxing_full)), test_size=0.6, stratify=shao_labels, random_state=42)
shao_train = torch.utils.data.Subset(shaoxing_full, train_idx)
shao_test = torch.utils.data.Subset(shaoxing_full, test_idx)
print(f"Shaoxing — Train: {len(shao_train)} (MI: {shao_labels[train_idx].sum()}) | Test: {len(shao_test)} (MI: {shao_labels[test_idx].sum()})")

PTB-XL — Train: 17418 | Val: 2183 | MI: 25.09%
Scanning labels for 45152 Shaoxing records...


Labels: 100%|██████████| 45152/45152 [02:04<00:00, 363.27it/s]

MI: 123 (0.27%)
Shaoxing — Train: 18060 (MI: 49) | Test: 27092 (MI: 74)


In [3]:
# Augmented PTB-XL
class AugmentedPTBXLDataset(Dataset):
    def __init__(self, df, base_path, lead_idx=0, augment=True):
        self.df = df
        self.base_path = base_path
        self.lead_idx = lead_idx
        self.augment = augment
    def __len__(self): return len(self.df)
    def __getitem__(self, idx):
        filename = os.path.join(self.base_path, self.df.iloc[idx].filename_lr)
        signal, _ = wfdb.rdsamp(filename)
        s = signal[:, self.lead_idx].astype(np.float32)
        if self.augment:
            if np.random.random() < 0.5: s += np.random.normal(0, 0.02, s.shape).astype(np.float32)
            if np.random.random() < 0.5: s *= np.random.uniform(0.8, 1.2)
            if np.random.random() < 0.3: s = np.roll(s, np.random.randint(-50, 50))
        s = (s - s.mean()) / (s.std() + 1e-8)
        return torch.tensor(s, dtype=torch.float32).unsqueeze(0), torch.tensor(self.df.iloc[idx].MI, dtype=torch.float32)

## 2. CNN-Transformer Architecture + Training Utilities

In [4]:
class CATNet(nn.Module):
    def __init__(self, num_classes=1, d_model=64, nhead=4, num_encoder_layers=2):
        super().__init__()
        self.cnn = nn.Sequential(
            nn.Conv1d(1, 32, kernel_size=7, stride=2, padding=3, bias=False),
            nn.BatchNorm1d(32), nn.ReLU(),
            nn.Conv1d(32, 64, kernel_size=5, stride=2, padding=2, bias=False),
            nn.BatchNorm1d(64), nn.ReLU(),
            nn.Conv1d(64, d_model, kernel_size=3, stride=2, padding=1, bias=False),
            nn.BatchNorm1d(d_model), nn.ReLU(),
        )
        self.pos_embed = nn.Parameter(torch.zeros(1, 125, d_model))
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=nhead, dim_feedforward=d_model*4,
            dropout=0.1, batch_first=True, activation='gelu')
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_encoder_layers)
        self.pool = nn.AdaptiveAvgPool1d(1)
        self.head = nn.Linear(d_model, num_classes)
    def forward(self, x):
        x = self.cnn(x).transpose(1, 2)
        x = x + self.pos_embed[:, :x.size(1), :]
        x = self.transformer(x).transpose(1, 2)
        return self.head(self.pool(x).squeeze(-1))

class FocalLoss(nn.Module):
    def __init__(self, alpha=0.25, gamma=2.0):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma
    def forward(self, logits, targets):
        bce = F.binary_cross_entropy_with_logits(logits, targets, reduction='none')
        pt = torch.exp(-bce)
        alpha_t = self.alpha * targets + (1 - self.alpha) * (1 - targets)
        return (alpha_t * (1 - pt) ** self.gamma * bce).mean()

def evaluate(loader, model, criterion, dev):
    model.eval()
    losses, preds, targets = [], [], []
    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(dev), y.to(dev)
            out = model(x).squeeze()
            if out.dim() == 0: out = out.unsqueeze(0)
            losses.append(criterion(out, y).item())
            preds.extend(torch.sigmoid(out).cpu().numpy())
            targets.extend(y.cpu().numpy())
    if len(set(targets)) < 2: return np.mean(losses), 0.5
    return np.mean(losses), roc_auc_score(targets, preds)

def train_epoch(loader, model, criterion, optimizer, dev):
    model.train()
    losses, preds, targets = [], [], []
    for x, y in loader:
        x, y = x.to(dev), y.to(dev)
        optimizer.zero_grad()
        out = model(x).squeeze()
        if out.dim() == 0: out = out.unsqueeze(0)
        loss = criterion(out, y)
        loss.backward()
        optimizer.step()
        losses.append(loss.item())
        preds.extend(torch.sigmoid(out).detach().cpu().numpy())
        targets.extend(y.detach().cpu().numpy())
    if len(set(targets)) < 2: return np.mean(losses), 0.5
    return np.mean(losses), roc_auc_score(targets, preds)

print(f"CNN-Transformer params: {sum(p.numel() for p in CATNet().parameters()):,}")

CNN-Transformer params: 131,105


## 3. Optuna Objective

In [5]:
def objective(trial, pruner_name):
    # --- Hyperparameters ---
    lr = trial.suggest_float("lr", 1e-5, 1e-2, log=True)
    warmup_epochs = trial.suggest_int("warmup_epochs", 0, 5)
    freeze_cnn = trial.suggest_categorical("freeze_cnn", [True, False])
    
    print(f"\n  Trial {trial.number}: lr={lr:.6f}, warmup={warmup_epochs}, freeze_cnn={freeze_cnn}")
    
    batch_size = 64
    criterion = FocalLoss(alpha=0.25, gamma=2.0)
    
    # --- Dataloaders ---
    train_loader = DataLoader(AugmentedPTBXLDataset(train_df, ptb_path, augment=True), batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(AugmentedPTBXLDataset(val_df, ptb_path, augment=False), batch_size=batch_size)
    
    # --- Pre-training ---
    model = CATNet().to(device)
    optimizer = optim.Adam(model.parameters(), lr=lr)
    
    pretrain_max = 100
    def lr_lambda(epoch):
        if epoch < warmup_epochs:
            return (epoch + 1) / max(warmup_epochs, 1)
        cos_epoch = epoch - warmup_epochs
        cos_max = pretrain_max - warmup_epochs
        return 0.5 * (1 + np.cos(np.pi * cos_epoch / max(cos_max, 1)))
    scheduler = optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)
    
    best_pretrain_auc = 0
    patience_counter = 0
    pretrain_epochs_run = 0
    
    for epoch in range(pretrain_max):
        train_epoch(train_loader, model, criterion, optimizer, device)
        _, v_auc = evaluate(val_loader, model, criterion, device)
        scheduler.step()
        pretrain_epochs_run = epoch + 1
        
        if v_auc > best_pretrain_auc:
            best_pretrain_auc = v_auc
            patience_counter = 0
            torch.save({'model_state_dict': model.state_dict()},
                       f"../checkpoints/catnet_tuning_trial{trial.number}_pretrain.pt")
        else:
            patience_counter += 1
        
        # Print every 5 epochs
        if (epoch + 1) % 5 == 0 or epoch == 0:
            print(f"    Pretrain {epoch+1:3d}/{pretrain_max}: V-AUC {v_auc:.4f} (best: {best_pretrain_auc:.4f}, patience: {patience_counter})")
        
        trial.report(v_auc, epoch)
        if trial.should_prune():
            print(f"    PRUNED at pretrain epoch {epoch+1}")
            db.execute("INSERT INTO catnet_tuning VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)",
                [pruner_name, trial.number, datetime.now(), pruner_name,
                 lr, warmup_epochs, freeze_cnn,
                 best_pretrain_auc, pretrain_epochs_run, None, None, None, 'pruned_pretrain'])
            raise optuna.TrialPruned()
        
        if patience_counter >= 15:
            print(f"    Pretrain early stop at epoch {epoch+1}")
            break
    
    print(f"    Pretrain done: best V-AUC {best_pretrain_auc:.4f} in {pretrain_epochs_run} epochs")
    
    # Load best pretrained
    ckpt = torch.load(f"../checkpoints/catnet_tuning_trial{trial.number}_pretrain.pt",
                      map_location=device, weights_only=False)
    model.load_state_dict(ckpt['model_state_dict'])
    
    # --- Zero-shot ---
    shao_full_loader = DataLoader(shaoxing_full, batch_size=batch_size)
    _, zeroshot_auc = evaluate(shao_full_loader, model, criterion, device)
    print(f"    Zero-shot AUC: {zeroshot_auc:.4f}")
    
    # --- Fine-tuning ---
    if freeze_cnn:
        for param in model.cnn.parameters():
            param.requires_grad = False
        ft_params = [p for p in model.parameters() if p.requires_grad]
    else:
        for param in model.parameters():
            param.requires_grad = True
        ft_params = model.parameters()
    
    ft_lr = lr * 0.1
    ft_optimizer = optim.Adam(ft_params, lr=ft_lr)
    
    finetune_max = 50
    def ft_lr_lambda(epoch):
        if epoch < warmup_epochs:
            return (epoch + 1) / max(warmup_epochs, 1)
        cos_epoch = epoch - warmup_epochs
        cos_max = finetune_max - warmup_epochs
        return 0.5 * (1 + np.cos(np.pi * cos_epoch / max(cos_max, 1)))
    ft_scheduler = optim.lr_scheduler.LambdaLR(ft_optimizer, ft_lr_lambda)
    
    shao_train_loader = DataLoader(shao_train, batch_size=batch_size, shuffle=True)
    shao_test_loader = DataLoader(shao_test, batch_size=batch_size)
    
    best_ft_auc = 0
    ft_patience = 0
    finetune_epochs_run = 0
    
    for epoch in range(finetune_max):
        train_epoch(shao_train_loader, model, criterion, ft_optimizer, device)
        _, v_auc = evaluate(shao_test_loader, model, criterion, device)
        ft_scheduler.step()
        finetune_epochs_run = epoch + 1
        
        if v_auc > best_ft_auc:
            best_ft_auc = v_auc
            ft_patience = 0
            torch.save({'model_state_dict': model.state_dict()},
                       f"../checkpoints/catnet_tuning_trial{trial.number}_finetuned.pt")
        else:
            ft_patience += 1
        
        # Print every 5 epochs
        if (epoch + 1) % 5 == 0 or epoch == 0:
            print(f"    Finetune {epoch+1:3d}/{finetune_max}: V-AUC {v_auc:.4f} (best: {best_ft_auc:.4f}, patience: {ft_patience})")
        
        trial.report(v_auc, pretrain_max + epoch)
        if trial.should_prune():
            print(f"    PRUNED at finetune epoch {epoch+1}")
            db.execute("INSERT INTO catnet_tuning VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)",
                [pruner_name, trial.number, datetime.now(), pruner_name,
                 lr, warmup_epochs, freeze_cnn,
                 best_pretrain_auc, pretrain_epochs_run, zeroshot_auc,
                 best_ft_auc, finetune_epochs_run, 'pruned_finetune'])
            raise optuna.TrialPruned()
        
        if ft_patience >= 10:
            print(f"    Finetune early stop at epoch {epoch+1}")
            break
    
    # Log completed trial
    db.execute("INSERT INTO catnet_tuning VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)",
        [pruner_name, trial.number, datetime.now(), pruner_name,
         lr, warmup_epochs, freeze_cnn,
         best_pretrain_auc, pretrain_epochs_run, zeroshot_auc,
         best_ft_auc, finetune_epochs_run, 'completed'])
    
    print(f"  => Trial {trial.number} DONE: pretrain={best_pretrain_auc:.4f}, zeroshot={zeroshot_auc:.4f}, finetune={best_ft_auc:.4f}")
    
    return best_ft_auc


## 4. Run Optuna Studies

In [6]:
N_TRIALS = 15

# Study 1: MedianPruner
print("=" * 60)
print("  Study 1: MedianPruner")
print("=" * 60)
study_median = optuna.create_study(
    direction="maximize",
    pruner=MedianPruner(n_startup_trials=3, n_warmup_steps=10),
    study_name="catnet_median"
)
study_median.optimize(lambda trial: objective(trial, "median"), n_trials=N_TRIALS)

print(f"\nMedianPruner best: {study_median.best_value:.4f}")
print(f"  Params: {study_median.best_params}")

[I 2026-04-03 21:26:45,993] A new study created in memory with name: catnet_median


  Study 1: MedianPruner

  Trial 0: lr=0.001615, warmup=5, freeze_cnn=True
    Pretrain   1/100: V-AUC 0.7658 (best: 0.7658, patience: 0)
    Pretrain   5/100: V-AUC 0.7769 (best: 0.7833, patience: 1)
    Pretrain  10/100: V-AUC 0.7839 (best: 0.7863, patience: 1)
    Pretrain  15/100: V-AUC 0.7924 (best: 0.7925, patience: 1)
    Pretrain  20/100: V-AUC 0.7892 (best: 0.7925, patience: 6)
    Pretrain  25/100: V-AUC 0.7818 (best: 0.7925, patience: 11)
    Pretrain early stop at epoch 29
    Pretrain done: best V-AUC 0.7925 in 29 epochs
    Zero-shot AUC: 0.8152
    Finetune   1/50: V-AUC 0.7883 (best: 0.7883, patience: 0)
    Finetune   5/50: V-AUC 0.8242 (best: 0.8242, patience: 0)
    Finetune  10/50: V-AUC 0.8154 (best: 0.8297, patience: 3)
    Finetune  15/50: V-AUC 0.7726 (best: 0.8297, patience: 8)


[I 2026-04-03 22:35:20,097] Trial 0 finished with value: 0.8296796129907389 and parameters: {'lr': 0.0016145119209494295, 'warmup_epochs': 5, 'freeze_cnn': True}. Best is trial 0 with value: 0.8296796129907389.


    Finetune early stop at epoch 17
  => Trial 0 DONE: pretrain=0.7925, zeroshot=0.8152, finetune=0.8297

  Trial 1: lr=0.000038, warmup=0, freeze_cnn=False
    Pretrain   1/100: V-AUC 0.7200 (best: 0.7200, patience: 0)
    Pretrain   5/100: V-AUC 0.7675 (best: 0.7675, patience: 0)
    Pretrain  10/100: V-AUC 0.7802 (best: 0.7802, patience: 0)
    Pretrain  15/100: V-AUC 0.7805 (best: 0.7829, patience: 2)
    Pretrain  20/100: V-AUC 0.7805 (best: 0.7840, patience: 3)
    Pretrain  25/100: V-AUC 0.7828 (best: 0.7843, patience: 4)
    Pretrain  30/100: V-AUC 0.7853 (best: 0.7853, patience: 0)
    Pretrain  35/100: V-AUC 0.7805 (best: 0.7853, patience: 5)
    Pretrain  40/100: V-AUC 0.7842 (best: 0.7867, patience: 1)
    Pretrain  45/100: V-AUC 0.7832 (best: 0.7867, patience: 6)
    Pretrain  50/100: V-AUC 0.7817 (best: 0.7867, patience: 11)
    Pretrain early stop at epoch 54
    Pretrain done: best V-AUC 0.7867 in 54 epochs
    Zero-shot AUC: 0.7983
    Finetune   1/50: V-AUC 0.7721 (be

[I 2026-04-04 01:24:11,035] Trial 1 finished with value: 0.7974651033445171 and parameters: {'lr': 3.778104895053396e-05, 'warmup_epochs': 0, 'freeze_cnn': False}. Best is trial 0 with value: 0.8296796129907389.


    Finetune  50/50: V-AUC 0.7975 (best: 0.7975, patience: 0)
  => Trial 1 DONE: pretrain=0.7867, zeroshot=0.7983, finetune=0.7975

  Trial 2: lr=0.009396, warmup=2, freeze_cnn=True
    Pretrain   1/100: V-AUC 0.7294 (best: 0.7294, patience: 0)
    Pretrain   5/100: V-AUC 0.5740 (best: 0.7294, patience: 4)
    Pretrain  10/100: V-AUC 0.5362 (best: 0.7294, patience: 9)
    Pretrain  15/100: V-AUC 0.5111 (best: 0.7294, patience: 14)
    Pretrain early stop at epoch 16
    Pretrain done: best V-AUC 0.7294 in 16 epochs
    Zero-shot AUC: 0.7073
    Finetune   1/50: V-AUC 0.7281 (best: 0.7281, patience: 0)
    Finetune   5/50: V-AUC 0.7300 (best: 0.7646, patience: 1)
    Finetune  10/50: V-AUC 0.6945 (best: 0.7646, patience: 6)


... (22 outputs left out)

[I 2026-04-04 08:58:32,241] Trial 13 pruned. 


    PRUNED at pretrain epoch 11

  Trial 14: lr=0.000183, warmup=3, freeze_cnn=True
    Pretrain   1/100: V-AUC 0.7381 (best: 0.7381, patience: 0)
    Pretrain   5/100: V-AUC 0.7760 (best: 0.7760, patience: 0)
    Pretrain  10/100: V-AUC 0.7850 (best: 0.7850, patience: 0)


[I 2026-04-04 09:11:26,851] Trial 14 pruned. 


    PRUNED at pretrain epoch 12

MedianPruner best: 0.8344
  Params: {'lr': 0.0004923850117591977, 'warmup_epochs': 1, 'freeze_cnn': True}


In [7]:
# Study 2: HyperbandPruner
print("=" * 60)
print("  Study 2: HyperbandPruner")
print("=" * 60)
study_hyperband = optuna.create_study(
    direction="maximize",
    pruner=HyperbandPruner(min_resource=10, max_resource=150, reduction_factor=3),
    study_name="catnet_hyperband"
)
study_hyperband.optimize(lambda trial: objective(trial, "hyperband"), n_trials=N_TRIALS)

print(f"\nHyperbandPruner best: {study_hyperband.best_value:.4f}")
print(f"  Params: {study_hyperband.best_params}")

[I 2026-04-04 09:11:26,858] A new study created in memory with name: catnet_hyperband


  Study 2: HyperbandPruner

  Trial 0: lr=0.000356, warmup=5, freeze_cnn=False
    Pretrain   1/100: V-AUC 0.7403 (best: 0.7403, patience: 0)
    Pretrain   5/100: V-AUC 0.7841 (best: 0.7841, patience: 0)
    Pretrain  10/100: V-AUC 0.7909 (best: 0.7909, patience: 0)
    Pretrain  15/100: V-AUC 0.7869 (best: 0.7909, patience: 5)
    Pretrain  20/100: V-AUC 0.7852 (best: 0.7929, patience: 1)
    Pretrain  25/100: V-AUC 0.7819 (best: 0.7929, patience: 6)
    Pretrain  30/100: V-AUC 0.7823 (best: 0.7929, patience: 11)
    Pretrain early stop at epoch 34
    Pretrain done: best V-AUC 0.7929 in 34 epochs
    Zero-shot AUC: 0.8229
    Finetune   1/50: V-AUC 0.8050 (best: 0.8050, patience: 0)
    Finetune   5/50: V-AUC 0.8271 (best: 0.8271, patience: 0)
    Finetune  10/50: V-AUC 0.8515 (best: 0.8530, patience: 2)
    Finetune  15/50: V-AUC 0.8268 (best: 0.8530, patience: 7)


[I 2026-04-04 10:43:57,010] Trial 0 finished with value: 0.8529966508813944 and parameters: {'lr': 0.0003564021067757964, 'warmup_epochs': 5, 'freeze_cnn': False}. Best is trial 0 with value: 0.8529966508813944.


    Finetune early stop at epoch 18
  => Trial 0 DONE: pretrain=0.7929, zeroshot=0.8229, finetune=0.8530

  Trial 1: lr=0.001240, warmup=2, freeze_cnn=False
    Pretrain   1/100: V-AUC 0.7617 (best: 0.7617, patience: 0)
    Pretrain   5/100: V-AUC 0.7816 (best: 0.7816, patience: 0)
    Pretrain  10/100: V-AUC 0.7839 (best: 0.7856, patience: 3)
    Pretrain  15/100: V-AUC 0.7827 (best: 0.7900, patience: 3)
    Pretrain  20/100: V-AUC 0.7843 (best: 0.7966, patience: 3)
    Pretrain  25/100: V-AUC 0.7808 (best: 0.7966, patience: 8)
    Pretrain  30/100: V-AUC 0.7731 (best: 0.7966, patience: 13)


[I 2026-04-04 11:20:37,927] Trial 1 pruned. 


    PRUNED at pretrain epoch 31

  Trial 2: lr=0.000017, warmup=5, freeze_cnn=False
    Pretrain   1/100: V-AUC 0.5313 (best: 0.5313, patience: 0)
    Pretrain   5/100: V-AUC 0.7292 (best: 0.7292, patience: 0)
    Pretrain  10/100: V-AUC 0.7559 (best: 0.7571, patience: 1)


... (22 outputs left out)

[I 2026-04-04 21:25:55,787] Trial 13 pruned. 


    Finetune   1/50: V-AUC 0.8019 (best: 0.8019, patience: 0)
    PRUNED at finetune epoch 1

  Trial 14: lr=0.001911, warmup=5, freeze_cnn=True
    Pretrain   1/100: V-AUC 0.7633 (best: 0.7633, patience: 0)
    Pretrain   5/100: V-AUC 0.7794 (best: 0.7794, patience: 0)
    Pretrain  10/100: V-AUC 0.7853 (best: 0.7853, patience: 0)
    Pretrain  15/100: V-AUC 0.7797 (best: 0.7911, patience: 3)
    Pretrain  20/100: V-AUC 0.7818 (best: 0.7911, patience: 8)
    Pretrain  25/100: V-AUC 0.7773 (best: 0.7911, patience: 13)
    Pretrain early stop at epoch 27
    Pretrain done: best V-AUC 0.7911 in 27 epochs
    Zero-shot AUC: 0.8006


[I 2026-04-04 22:01:16,297] Trial 14 pruned. 


    Finetune   1/50: V-AUC 0.7802 (best: 0.7802, patience: 0)
    PRUNED at finetune epoch 1

HyperbandPruner best: 0.8530
  Params: {'lr': 0.0003564021067757964, 'warmup_epochs': 5, 'freeze_cnn': False}


## 5. Compare Pruners

In [8]:
print("=" * 60)
print("  Pruner Comparison")
print("=" * 60)

for name, study in [('MedianPruner', study_median), ('HyperbandPruner', study_hyperband)]:
    completed = [t for t in study.trials if t.state == optuna.trial.TrialState.COMPLETE]
    pruned = [t for t in study.trials if t.state == optuna.trial.TrialState.PRUNED]
    print(f"\n{name}:")
    print(f"  Completed: {len(completed)} | Pruned: {len(pruned)}")
    print(f"  Best AUC: {study.best_value:.4f}")
    print(f"  Best params: {study.best_params}")

# Pick overall winner
if study_median.best_value >= study_hyperband.best_value:
    best_study = study_median
    best_pruner = "MedianPruner"
else:
    best_study = study_hyperband
    best_pruner = "HyperbandPruner"

print(f"\nOverall best: {best_pruner} with AUC {best_study.best_value:.4f}")
print(f"  {best_study.best_params}")

  Pruner Comparison

MedianPruner:
  Completed: 6 | Pruned: 9
  Best AUC: 0.8344
  Best params: {'lr': 0.0004923850117591977, 'warmup_epochs': 1, 'freeze_cnn': True}

HyperbandPruner:
  Completed: 5 | Pruned: 10
  Best AUC: 0.8530
  Best params: {'lr': 0.0003564021067757964, 'warmup_epochs': 5, 'freeze_cnn': False}

Overall best: HyperbandPruner with AUC 0.8530
  {'lr': 0.0003564021067757964, 'warmup_epochs': 5, 'freeze_cnn': False}


## 6. DuckDB Experiment Log

In [9]:
print("=== All CNN-Transformer Tuning Trials ===")
print(db.execute("""
    SELECT pruner, trial_number, 
           ROUND(lr, 6) as lr, warmup_epochs, freeze_cnn,
           ROUND(pretrain_best_auc, 4) as pretrain_auc,
           pretrain_epochs_run as pt_epochs,
           ROUND(zeroshot_auc, 4) as zeroshot,
           ROUND(finetune_best_auc, 4) as finetune_auc,
           finetune_epochs_run as ft_epochs,
           status
    FROM catnet_tuning
    ORDER BY finetune_best_auc DESC NULLS LAST
""").df().to_string(index=False))

db.close()

=== All CNN-Transformer Tuning Trials ===
   pruner  trial_number       lr  warmup_epochs  freeze_cnn  pretrain_auc  pt_epochs  zeroshot  finetune_auc  ft_epochs          status
hyperband             0 0.000356              5       False        0.7929         34    0.8229        0.8530         18       completed
hyperband             7 0.001420              5       False        0.7907         24    0.8206        0.8355         18       completed
   median            12 0.000492              1        True        0.7919         37    0.8188        0.8344         21       completed
   median             0 0.001615              5        True        0.7925         29    0.8152        0.8297         17       completed
hyperband             5 0.007349              3       False        0.7600         17    0.7531        0.8276         15       completed
   median            10 0.000548              0        True        0.7919         25    0.8154        0.8211         21       completed
hyperb